# Notebook 04 — Household Structural Features (HUS) and Integration with the Person Sample (PUS)

Construct the household-level structural feature layer from ACS PUMS housing records (HUS) and integrate it into the person-level modeling sample derived from PUS data.

## Objectives

- Load HUS microdata and the person-level modeling sample generated in Notebook 03
- Select and standardize relevant household variables (lowercase, snake_case, consistent data types)
- Merge household features into the person sample using `serialno`
- Validate merge integrity
- Save the integrated structural modeling dataset for downstream clustering

In [2]:
# Imports and dataset loading

import re
from pathlib import Path

import numpy as np
import pandas as pd

# -------------------------------------------------------------------
# Project paths
# -------------------------------------------------------------------

# Project root (works when notebook is executed from /notebooks)
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"

ACS_PATH = INTERIM_DIR / "acs_pums_5y"

# -------------------------------------------------------------------
# Dataset files
# -------------------------------------------------------------------

PUS_SAMPLE_FILE = "us_structural_modeling_sample_v1.parquet"
HUS_FILE = "_consolidated_housing.parquet"

PUS_PATH = ACS_PATH / PUS_SAMPLE_FILE
HUS_PATH = ACS_PATH / HUS_FILE

if not PUS_PATH.exists():
    raise FileNotFoundError(
        "PUS modeling sample not found. Run notebook 03_acs_modeling_sample first."
    )

if not HUS_PATH.exists():
    raise FileNotFoundError(
        "Consolidated housing dataset not found. Run notebook 01_acs_pums_ingest first."
    )

# -------------------------------------------------------------------
# Load datasets
# -------------------------------------------------------------------

df_pus = pd.read_parquet(PUS_PATH)
df_hus = pd.read_parquet(HUS_PATH)

print("PUS sample:", df_pus.shape)
print("HUS:", df_hus.shape)

df_pus.head()

PUS sample: (1000000, 13)
HUS: (7650159, 11)


,serialno,sporder,pwgtp,actor_class,age_bin,sex_label,race_eth,edu_tier,emp_tier,income_tier_fixed,income_tier_pct,mar_tier,commute_tier
0,2023HU1043211,2,58,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P0-20,Never_Married,Car
1,2019HU1076190,2,46,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P20-40,Never_Married,Car
2,2019GQ0046130,1,12,Adult,35-44,Male,Black_NH,HS_or_less,Other_Not_in_Labor_Force,0-19k,P0-20,Never_Married,NaN
3,2019HU0403832,1,76,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P0-20,Never_Married,Work_From_Home
4,2019HU0277198,1,64,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P0-20,Never_Married,Car


In [3]:
# Step 1 — Inspect HUS schema

print("Columns:", df_hus.columns.tolist())

for col in df_hus.columns:
    print("\n", col)
    print(df_hus[col].value_counts().head())
    print("Unique:", df_hus[col].nunique())

Columns: ['SERIALNO', 'WGTP', 'TEN', 'HINCP', 'NP', 'BDSP', 'VEH', 'FINCP', 'GRNTP', 'SMOCP', 'PUMA']

 SERIALNO
SERIALNO
2019GQ0000088    1
2019GQ0000096    1
2019GQ0000153    1
2019GQ0000198    1
2019GQ0000205    1
Name: count, dtype: int64
Unique: 7650159

 WGTP
WGTP
0     854342
11    343154
12    333427
10    321520
13    303822
Name: count, dtype: int64
Unique: 529

 TEN
TEN
1.0    2578358
2.0    1929282
3.0    1612538
4.0     111968
Name: count, dtype: int64
Unique: 4

 HINCP
HINCP
0.0         71501
50000.0     51870
60000.0     51632
40000.0     46153
100000.0    46123
Name: count, dtype: int64
Unique: 51359

 NP
NP
1    2592137
2    2321169
3     915054
4     721617
0     563671
Name: count, dtype: int64
Unique: 21

 BDSP
BDSP
3.0    2794582
2.0    1616104
4.0    1259552
1.0     619067
5.0     301360
Name: count, dtype: int64
Unique: 9

 VEH
VEH
2.0    2375119
1.0    1952969
3.0     974630
0.0     438273
4.0     337867
Name: count, dtype: int64
Unique: 7

 FINCP
FINCP
0.0     

In [4]:
# Step 2 — Build HUS structural layer

df_hus_struct = df_hus[[
    "SERIALNO",
    "TEN",
    "HINCP",
    "NP",
    "VEH",
    "PUMA"
]].copy()

# snake_case
df_hus_struct.columns = [
    "serialno",
    "tenure",
    "household_income",
    "household_size",
    "vehicle_count",
    "puma"
]

# create household income tiers (same as PUS)
bins = [-np.inf, 19999, 49999, 99999, 199999, np.inf]
labels = ["0-19k", "20-49k", "50-99k", "100-199k", "200k+"]

df_hus_struct["hhincome_tier"] = pd.cut(
    df_hus_struct["household_income"],
    bins=bins,
    labels=labels
)

# drop raw income
df_hus_struct = df_hus_struct.drop(columns="household_income")

print(df_hus_struct.shape)
df_hus_struct.head()

(7650159, 6)


,serialno,tenure,household_size,vehicle_count,puma,hhincome_tier
0,2019GQ0000088,NaN,1,NaN,2600,NaN
1,2019GQ0000096,NaN,1,NaN,700,NaN
2,2019GQ0000153,NaN,1,NaN,800,NaN
3,2019GQ0000198,NaN,1,NaN,800,NaN
4,2019GQ0000205,NaN,1,NaN,2803,NaN


In [5]:
# Step 1 — inspect serial number prefixes

prefix_counts = (
    df_hus_struct["serialno"]
    .str.extract(r"([A-Z]{2})")   # first two letters
    .value_counts()
)

print(prefix_counts)

0 
HU    6795817
GQ     854342
Name: count, dtype: int64


In [6]:
# labeling 
df_hus_struct["household_type"] = np.where(
    df_hus_struct["serialno"].str.contains("GQ"),
    "group_quarters",
    "housing_unit"
)

In [7]:
df_hus_struct.info()

<class 'pandas.DataFrame'>
RangeIndex: 7650159 entries, 0 to 7650158
Data columns (total 7 columns):
 #   Column          Dtype   
---  ------          -----   
 0   serialno        str     
 1   tenure          float64 
 2   household_size  int64   
 3   vehicle_count   float64 
 4   puma            int64   
 5   hhincome_tier   category
 6   household_type  str     
dtypes: category(1), float64(2), int64(2), str(2)
memory usage: 541.5 MB


In [8]:
# Check missingness rates by household_type

cols = ["tenure", "hhincome_tier", "vehicle_count"]

for c in cols:
    print("\n", c)
    print(
        df_hus_struct
        .groupby("household_type")[c]
        .apply(lambda x: x.isna().mean())
    )


 tenure
household_type
group_quarters    1.000000
housing_unit      0.082944
Name: tenure, dtype: float64

 hhincome_tier
household_type
group_quarters    1.000000
housing_unit      0.082944
Name: hhincome_tier, dtype: float64

 vehicle_count
household_type
group_quarters    1.000000
housing_unit      0.082944
Name: vehicle_count, dtype: float64


In [9]:
# Step — Cast dtypes to allow explicit GQ labels, then fill for GQ only

gq_mask = df_hus_struct["household_type"] == "group_quarters"

# tenure: make it string so it can hold "group_quarters" (keep HU missing as <NA>)
df_hus_struct["tenure"] = df_hus_struct["tenure"].astype("Int64").astype("string")

# hhincome_tier: ensure string (keeps HU missing as <NA>)
df_hus_struct["hhincome_tier"] = df_hus_struct["hhincome_tier"].astype("string")

# vehicle_count: keep numeric, but ensure it can hold NA + ints
df_hus_struct["vehicle_count"] = df_hus_struct["vehicle_count"].astype("Int64")

# Apply structural replacements ONLY for group quarters
df_hus_struct.loc[gq_mask, "tenure"] = "group_quarters"
df_hus_struct.loc[gq_mask, "hhincome_tier"] = "group_quarters"
df_hus_struct.loc[gq_mask, "vehicle_count"] = 0

In [10]:
df_hus_struct.groupby("household_type")[["tenure","hhincome_tier","vehicle_count"]].apply(
    lambda x: x.isna().mean()
)

,tenure,hhincome_tier,vehicle_count
household_type,,,
group_quarters,0.000000,0.000000,0.000000
housing_unit,0.082944,0.082944,0.082944


In [11]:
df_hus_struct.head()

,serialno,tenure,household_size,vehicle_count,puma,hhincome_tier,household_type
0,2019GQ0000088,group_quarters,1,0,2600,group_quarters,group_quarters
1,2019GQ0000096,group_quarters,1,0,700,group_quarters,group_quarters
2,2019GQ0000153,group_quarters,1,0,800,group_quarters,group_quarters
3,2019GQ0000198,group_quarters,1,0,800,group_quarters,group_quarters
4,2019GQ0000205,group_quarters,1,0,2803,group_quarters,group_quarters


In [12]:
# Step — Validate HUS uniqueness

assert df_hus_struct["serialno"].is_unique, "HUS contains duplicate households!"

In [13]:
# Step — Merge PUS sample with HUS features

df_model = df_pus.merge(
    df_hus_struct,
    on="serialno",
    how="left",
    validate="many_to_one"
)

print("PUS rows:", len(df_pus))
print("Merged rows:", len(df_model))

PUS rows: 1000000
Merged rows: 1000000


In [14]:
# Check how many persons received household attributes

merge_missing = df_model["household_type"].isna().mean()

print("Rows missing HUS info:", merge_missing)

Rows missing HUS info: 0.0


In [15]:
df_model.head()

,serialno,sporder,pwgtp,actor_class,age_bin,sex_label,race_eth,edu_tier,emp_tier,income_tier_fixed,income_tier_pct,mar_tier,commute_tier,tenure,household_size,vehicle_count,puma,hhincome_tier,household_type
0,2023HU1043211,2,58,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P0-20,Never_Married,Car,3,2,0,4316,0-19k,housing_unit
1,2019HU1076190,2,46,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P20-40,Never_Married,Car,3,4,2,5922,20-49k,housing_unit
2,2019GQ0046130,1,12,Adult,35-44,Male,Black_NH,HS_or_less,Other_Not_in_Labor_Force,0-19k,P0-20,Never_Married,NaN,group_quarters,1,0,11300,group_quarters,group_quarters
3,2019HU0403832,1,76,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P0-20,Never_Married,Work_From_Home,1,5,2,2510,50-99k,housing_unit
4,2019HU0277198,1,64,Adult,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,P0-20,Never_Married,Car,3,4,1,4607,20-49k,housing_unit


In [16]:
df_model.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 19 columns):
 #   Column             Non-Null Count    Dtype   
---  ------             --------------    -----   
 0   serialno           1000000 non-null  str     
 1   sporder            1000000 non-null  int64   
 2   pwgtp              1000000 non-null  int64   
 3   actor_class        1000000 non-null  str     
 4   age_bin            1000000 non-null  string  
 5   sex_label          1000000 non-null  string  
 6   race_eth           1000000 non-null  string  
 7   edu_tier           1000000 non-null  string  
 8   emp_tier           804465 non-null   str     
 9   income_tier_fixed  1000000 non-null  string  
 10  income_tier_pct    778466 non-null   str     
 11  mar_tier           1000000 non-null  string  
 12  commute_tier       473674 non-null   category
 13  tenure             1000000 non-null  string  
 14  household_size     1000000 non-null  int64   
 15  vehicle_count      1000000 

In [18]:
# Save integrated structural dataset

OUTPUT_FILE = ACS_PATH / "us_structural_population_v1.parquet"

df_model.to_parquet(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("Rows:", f"{len(df_model):,}")

Saved: /Users/marcomagnolo/Projects/us-audience-segmentation/data/interim/acs_pums_5y/us_structural_population_v1.parquet
Rows: 1,000,000
